# Testing the square root of the fission FEM matrix with zero fission cross sections

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, coo_matrix
import scipy as sp
import math
import itertools
import FEM_BPX_helpers as FEM

Test the creation of the mass matrix

In [4]:
D = 2 # dimensions
L = 3

# cross sections
# data I have been using: R=10. D1=0.1, D2=200, sigma_a_1=0.8, sigma_a_2=0.5, sigma_f_1=0.9, sigma_f_2=0.1
R = 1 # range of problem (in every dimension)
D_1 = 1.0 # diffusion coefficient in region 1 (1/cm)
D_2 = 10.0 # diffusion coefficient in region 2 (1/cm)
sigma_a_1 = 1.0 # macroscopic absorption cross section in region 1 (1/cm)
sigma_a_2 = 2.0 # macroscopic absorption cross section in region 2 (1/cm)
sigma_f_1 = 0.0 # macroscopic nu*fission cross section in region 1 (1/cm)
sigma_f_2 = 2.0 # macroscopic nu*fission cross section in region 2 (1/cm)
#diffusion_mat_base = np.array([D_1, D_2]) # 1D, mat_L=1
#absorption_xs_base = np.array([sigma_a_1, sigma_a_2]) # 1D, mat_L=1
#nu_fission_xs_base = np.array([sigma_f_1, sigma_f_2]) # 1D, mat_L=1
diffusion_mat_base = np.array([[D_1, D_2],[D_2, D_1]]) # 2D, mat_L=1
absorption_xs_base = np.array([[sigma_a_1, sigma_a_2],[sigma_a_2, sigma_a_1]]) # 2D, mat_L=1
nu_fission_xs_base = np.array([[sigma_f_1, sigma_f_2],[sigma_f_2, sigma_f_1]]) # 2D, mat_L=1
#diffusion_mat_base = np.array([[[D_1, D_2],[D_2, D_1]],[[D_2, D_1],[D_1, D_2]]]) # 3D, mat_L=1
#absorption_xs_base = np.array([[[sigma_a_1, sigma_a_2],[sigma_a_2, sigma_a_1]],[[sigma_a_2, sigma_a_1],[sigma_a_1, sigma_a_2]]]) # 3D, mat_L=1
#nu_fission_xs_base = np.array([[[sigma_f_1, sigma_f_2],[sigma_f_2, sigma_f_1]],[[sigma_f_2, sigma_f_1],[sigma_f_1, sigma_f_2]]]) # 3D, mat_L=1

# set up the 2D matrices of material properties (diffusion coefs and cross sections)
mat_L = 1 # the number of checkerboard spaces is 2^(D*mat_L)
'''if mat_L == 0:
    diffusion_mat = np.array([[D_1]])
    absorption_xs = np.array([[sigma_a_1]])
    nu_fission_xs = np.array([[sigma_f_1]])
else:
    diffusion_mat = np.kron(np.ones((2**(mat_L-1))), diffusion_mat_base)
    absorption_xs = np.kron(np.ones((2**(mat_L-1))), absorption_xs_base)
    nu_fission_xs = np.kron(np.ones((2**(mat_L-1))), nu_fission_xs_base)'''
if mat_L == 0:
    diffusion_mat = np.array([[D_1]])
    absorption_xs = np.array([[sigma_a_1]])
    nu_fission_xs = np.array([[sigma_f_1]])
else:
    diffusion_mat = np.kron(np.ones((2**(mat_L-1),2**(mat_L-1))), diffusion_mat_base) # 2D expansion of material grid
    absorption_xs = np.kron(np.ones((2**(mat_L-1),2**(mat_L-1))), absorption_xs_base)
    nu_fission_xs = np.kron(np.ones((2**(mat_L-1),2**(mat_L-1))), nu_fission_xs_base)
'''if mat_L == 0:
    diffusion_mat = np.array([[D_1]])
    absorption_xs = np.array([[sigma_a_1]])
    nu_fission_xs = np.array([[sigma_f_1]])
else:
    diffusion_mat = np.kron(np.ones((2**(mat_L-1),2**(mat_L-1),2**(mat_L-1))), diffusion_mat_base) # 3D expansion of material grid
    absorption_xs = np.kron(np.ones((2**(mat_L-1),2**(mat_L-1),2**(mat_L-1))), absorption_xs_base)
    nu_fission_xs = np.kron(np.ones((2**(mat_L-1),2**(mat_L-1),2**(mat_L-1))), nu_fission_xs_base)'''

N_1D = int(2**(L))
h = R/N_1D

N_1D = int(2**(L))
h = R/N_1D
nu_fission_xs_fine = np.kron(nu_fission_xs, np.ones((int(2**(L-mat_L)),) * D))
C_mat = h * FEM.get_2D_mass_matrix_LCU(D, L, nu_fission_xs_fine)
C_mat_numpy = C_mat.toarray()
C_mat_sqrt = sp.linalg.sqrtm(C_mat_numpy)

A_mat = FEM.get_mass_matrix_LCU(D, L, mat_L, absorption_xs)

# make the projector that selects the basis functions in B_1 of the 2D C matrix
B_0_indices = []
B_1_indices = [] # list of FEM basis function indices whose support region contains a non-zero fission cross section
for y_i in range(N_1D - 1):
    for x_i in range(N_1D - 1):
        if (nu_fission_xs_fine[x_i,y_i] == 0) and (nu_fission_xs_fine[x_i+1,y_i] == 0) and (nu_fission_xs_fine[x_i,y_i+1] == 0) and (nu_fission_xs_fine[x_i,y_i+1] == 0):
            B_0_indices.append(y_i * (N_1D - 1) + x_i)
        else:
            B_1_indices.append(y_i * (N_1D - 1) + x_i)
Pi_C = np.zeros(((N_1D - 1)**2, (N_1D - 1)**2))
for index in B_1_indices:
    Pi_C[index, index] = 1

Pi_0 = np.zeros(((N_1D - 1)**2, (N_1D - 1)**2))
for index in B_0_indices:
    Pi_0[index, index] = 1

C_fis = np.transpose(Pi_C) @ C_mat_numpy @ Pi_C
C_p = C_mat_numpy + Pi_0 @ np.eye((N_1D - 1)**2) @ Pi_0
C_p_sqrt = sp.linalg.sqrtm(C_p)
C_sqrt_qc = Pi_C @ C_p_sqrt @ Pi_C

C_sqrt_error = C_sqrt_qc - C_mat_sqrt
C_sqrt_scalar_error = np.linalg.norm(C_sqrt_error) # should be zero if this strategy works
print("done")

done
